# Demo 03: Post-training EpiZoo for a new species

This notebook demonstrates how to post-train **EpiZoo** for a new species.

Starting from a pretrained EpiZoo model, we perform species-specific post-training.

This demo illustrates the capability of EpiZoo for efficient cross-species adaptation without requiring training from scratch.

## Required files

Before running this notebook, please prepare the following files:

- **`seam.pth`**: SEAM checkpoint. (The SEAM checkpoint can be downloaded at [SEAM.pth](https://drive.google.com/file/d/1VlPnwrvMxkvKkqEOvM_pciCwJH5Jgb8Y/view?usp=drive_link))
- **`genome.fa`**: The reference genome sequence file of the new species.

## Output

This notebook will generate:

- Post-trained EpiZoo model checkpoint for the target species

In [1]:
import os
import sys

# Add EpiZoo root directory
PROJECT_ROOT = os.path.abspath("../")

if PROJECT_ROOT not in sys.path:
    sys.path.append(PROJECT_ROOT)

## Step 1: Compute sequence embeddings using SEAM

Before post-training, we first use SEAM from the pretrained EpiZoo to compute sequence embeddings for the cCREs of the new species. These sequence embeddings provide regulatory prior for model adaptation.

In [2]:
import scanpy as sc
from epizoo.data.ccre import extract_dna_sequences
from epizoo.data.datasets import SEAMDataset, collate_fn_seam
from torch.utils.data import DataLoader

# Load the dataset
adata_file_path = "../data/zebrafish_downsampled_2000_cells.h5ad"
adata = sc.read_h5ad(adata_file_path)
print(f"Anndata: {adata}")

# Load the reference genome
genome_file_path = "/data/lizhen/epizoo/genomes_fa/danRer11.fa"

# Extract DNA sequences for the regions in the dataset
regions = adata.var_names.tolist()
sequences = extract_dna_sequences(fasta_path=genome_file_path, regions=regions)
print(
    f"Example:\n"
    f"The first extracted DNA sequence length: {len(sequences[0])}\n"
    f"The first extracted DNA sequence:\n{sequences[0]}"
)

# Create SEAMDataset and dataloader
seamdataset = SEAMDataset(
    sequences=sequences,
    cfg_path="../config/SEAM_config",
)
dataloader = DataLoader(
    seamdataset,
    batch_size=128,
    shuffle=False,
    collate_fn=collate_fn_seam,
)

/home/jiangqun/miniconda3/envs/cellemu/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Anndata: AnnData object with n_obs × n_vars = 2000 × 466307
    obs: 'barcode', 'Cell_type', 'lineage', 'FRIP', 'dataset', 'nFrags', 'TSSEnrichment'


Extracting DNA sequences: 100%|██████████| 466307/466307 [00:04<00:00, 96831.19it/s]

Example:
The first extracted DNA sequence length: 500
The first extracted DNA sequence:
ATTCATGCTTTCATTACCTCTCGGTTAGACTACTGTAATGAGTTACTAGGCGGATGCCCTGCCGGCCTCACTAACAAACTGCAGCTGGTTCAAAATGCAGCTGCTGGAGTGCTTTCTAGAACTAAGAAATCACAGTACTCTAGTTCTTTCATCGGTGCATTGACTCCCATTAAATATTTAATTTACAATGTTAGTGTTTAAGTACAAAACCCTCAATGCTTTAGCTGCCCAGTATTTGAGGGAGCTCCTGGTGTATTATAGAGCATGAGATTGATTCTTTTAAGACCCTAAAGTGACCCACGAGTCTACATATTGATCCTGAAGGGTCAGTCTGTACACCAGCCGGTGACCTCTCATCTACAGCTTCTCCACAATGGATGTCCACCATTCTCCAGTCTCCGGTGCCTAGACTGCAGCTCTGTACAGTAAGTCTGGCCAGAGGAGAAATGGTCATGCCCAACTGAGCCTGGTTTGGAGGTTATTTTTCCCCTTTACTTTGG


In [3]:
import torch
from epizoo.models.seam import SEAMConfig, SEAM

# Load pretrained model
model_path = "/data/lizhen/epizoo/models/seam.pth"
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")

# Initialize SEAM model
config = SEAMConfig(cfg_path="../config/SEAM_config")
seam_model = SEAM(cfg=config)

# Load pretrained weights
state_dict = torch.load(model_path, map_location="cpu")

# Load SEAM
seam_model.load_state_dict(state_dict)
seam_model = seam_model.to(device)

print("SEAM model loaded successfully.")

SEAM model loaded successfully.


In [4]:
from epizoo.inference.embeddings import extract_seq_embeddings

# Compute sequence embeddings
sequence_embeddings = extract_seq_embeddings(
    seam_model,
    dataloader,
    device=device,
)
print(f"Completed. Sequence embeddings shape: {sequence_embeddings.shape}")

Extracting sequence embeddings: 100%|██████████| 3644/3644 [05:15<00:00, 11.56it/s]


Completed. Sequence embeddings shape: (466307, 512)


## Step 2: Model initialization

We initialize the cross-species model using the pretrained EpiZoo parameters and the sequence embeddings computed by SEAM in Step 1.

In [5]:
from epizoo.models.transfer import transfer_epizoox_state_dict
from epizoo.models.epizoo_x import EpiZooXConfig, EpiZooX

# Load pretrained weights
pretrained_model_path = "/data/lizhen/epizoo/models/pretrained_EpiZoo.pth"
pretrained_state_dict = torch.load(pretrained_model_path, map_location="cpu")

# Transfer weights
state_dict = transfer_epizoox_state_dict(
    state_dict=pretrained_state_dict,
    seq_embeddings=sequence_embeddings,
    num_ccres=adata.n_vars,
)

# Initialize EpiZoo-X model
config = EpiZooXConfig(
    vocab_size=adata.n_vars + 4,
    num_layers=30,
)
model = EpiZooX(cfg=config)

# Load the transferred weights
model.load_state_dict(state_dict, strict=False)

# Move model to device
model = model.to(device)

print("EpiZoo model loaded successfully.")

EpiZoo model loaded successfully.


## Step 3: Data processing

To prepare dataset for EpiZoo, we perform the following preprocessing steps:

1. **TF-IDF transformation:** Convert the raw count matrix into TF-IDF normalized values, which quantify the relative importance of accessible cCREs within each cell.

2. **Cell sentence generation:** Rank accessible cCREs according to their TF-IDF scores and convert each cell into a compact cell sentence composed of cCRE indices, which serves as the input for EpiZoo.

In [6]:
from epizoo.data.processing import compute_tfidf, generate_cell_sentences

# Perform TF-IDF transformation
adata = compute_tfidf(adata)

# Generate cell sentences
adata = generate_cell_sentences(adata)
print(f"Cell Sentences:\n{adata.obs['cell_indices'][:5]}")

TF-IDF completed. TF-IDF matrix stored in adata.X
Matrix shape: (2000, 466307)
Matrix type: <class 'scipy.sparse._csr.csr_matrix'>
Data type: float32
Non-zero entries: 2,970,369
Sparsity: 99.6815%
Non-zero value min: 1.468467
Non-zero value max: 147.622971
Non-zero value mean: 31.088402
Non-zero value median: 27.289757
--------------------------------------------------
Accessible cCREs per cell:
  Mean: 1485.18
  Median: 1168.00
  Min: 468
  Max: 11048
Cell Sentences:
batch2_56#56_GGCGACTAGTACGCCGTTCA    [287931, 289605, 122986, 282967, 438816, 20015...
batch2_75#75_TTAATGAGCTCATTCGACGG    [445416, 32331, 144418, 139459, 153027, 161858...
batch2_91#91_CGGATAAGCTTACTTACTTA    [234047, 128447, 365069, 122366, 361733, 41513...
batch2_53#53_AACGTAATCTACGCGAGATT    [463956, 454775, 34177, 192828, 330207, 330039...
batch2_56#56_GCTATCCTCCTGCATATAAC    [425879, 433922, 232136, 62687, 263783, 275440...
Name: cell_indices, dtype: object


## Step 4: Create Dataset and DataLoader

After generating cell sentences, we construct `CellDatasetX` and DataLoader. Each cell is represented as a sequence of tokens with special tokens [CLS] and [SEP].

In [7]:
from epizoo.data.datasets import CellDatasetX, collate_fn_x
from torch.utils.data import DataLoader

# Create a dataset and dataloader
celldatasetx = CellDatasetX(
    cell_sentences=adata.obs['cell_indices'].values,
    num_ccres=adata.n_vars,
)
dataloader = DataLoader(
    celldatasetx,
    batch_size=4,
    shuffle=True,
    collate_fn=collate_fn_x,
)

## Step 4: Model post-training

We post-train the transferred EpiZoo model on the scATAC-seq dataset of target species using `EpiZooXPostTrainer`.

The post-trained model checkpoint will be saved for downstream inference.

In [ ]:
from epizoo.train.posttrain import EpiZooXPostTrainer, EpiZooXPostTrainConfig

# Post-training configuration
ft_cfg = EpiZooXPostTrainConfig(
    mode="sr_cca",
    output_dir="/data/lizhen/epizoo/models/posttrained_model_for_zebrafish",
    max_steps=1000,
    save_steps=500,
    log_steps=100,
    keep_last=2,
    lr=5e-5,
    warmup_steps=200,
    device=device,
)

trainer = EpiZooXPostTrainer(
    model=model,
    train_loader=dataloader,
    cfg=ft_cfg,
)

# Start post-training
posttrained_model = trainer.train()

print(
    "This demo demonstrates the post-training workflow only. "
    "In practice, sufficient post-training steps are required."
)

Frozen seq_emb.
Step 100 | lr=2.500e-05 | loss=1.9445 | sr=0.9249 | cca=1.0196 | cca_pos_acc=0.3128 | cca_neg_acc=0.6874 | auroc=0.5032 | auprc=0.6016
Step 200 | lr=5.000e-05 | loss=1.5875 | sr=0.7553 | cca=0.8322 | cca_pos_acc=0.4310 | cca_neg_acc=0.5538 | auroc=0.4923 | auprc=0.5412
Step 300 | lr=5.000e-05 | loss=1.4040 | sr=0.6138 | cca=0.7902 | cca_pos_acc=0.4892 | cca_neg_acc=0.5451 | auroc=0.5206 | auprc=0.5403
Step 400 | lr=5.000e-05 | loss=1.3577 | sr=0.5871 | cca=0.7707 | cca_pos_acc=0.4764 | cca_neg_acc=0.5340 | auroc=0.5075 | auprc=0.5100
Step 500 | lr=4.500e-05 | loss=1.3497 | sr=0.5922 | cca=0.7575 | cca_pos_acc=0.4618 | cca_neg_acc=0.5366 | auroc=0.4999 | auprc=0.5656
Checkpoint saved to /data/lizhen/epizoo/models/posttrained_model_for_zebrafish/202607140836_500.pth
Epoch 1 finished.
Step 600 | lr=4.500e-05 | loss=1.3004 | sr=0.5515 | cca=0.7489 | cca_pos_acc=0.4578 | cca_neg_acc=0.5359 | auroc=0.4989 | auprc=0.6629
Step 700 | lr=4.500e-05 | loss=1.2829 | sr=0.5410 | cca=